# Lab 25-05 · Corners, Back Bias, Shuttles and Shrinks

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import foundry as f

## Part 1 · Corners and local variation

*Predict* the fraction of paths beyond the +45 mV slow corner with global σ = 15 mV and local σ = 10 mV.

In [2]:
dvt = f.simulate_vt(0.015, 0.010)
print(f"simulated {np.mean(dvt > 0.045):.4f}; analytical {f.fraction_beyond(0.045, 0.015, 0.010):.4f}")
d = f.delay(0.75, 0.30 + dvt) / f.delay(0.75, 0.30)
ss, ff = f.corner_ratio(0.75, 0.30, 0.0, 0.045)
plt.figure()
plt.hist(d, bins=100)
for x, lab in ((1.0, "TT"), (ss, "SS (Vt only)"), (ff, "FF (Vt only)")):
    plt.axvline(x, color="0.3", ls=":"); plt.text(x, plt.ylim()[1] * 0.9, lab, fontsize=7)
plt.xlabel("path delay / typical"); plt.ylabel("count"); plt.grid(True)
plt.show()

simulated 0.0065; analytical 0.0063


## Part 2 · Back-gate coupling

*Predict* the coupling with a 5 nm body.

In [3]:
for tsi in (7, 5):
    print(f"t_Si = {tsi} nm: formula {1000*f.coupling_formula(1.2, tsi, 20):.2f} mV/V, numerical {1000*f.coupling_numerical(1.2, tsi, 20):.2f} mV/V")
tb = np.linspace(5, 40, 50)
plt.figure()
plt.plot(tb, [1000 * f.coupling_formula(1.2, 7, t) for t in tb])
plt.xlabel("BOX thickness (nm)"); plt.ylabel("coupling (mV/V)"); plt.grid(True)
plt.show()

t_Si = 7 nm: formula 53.73 mV/V, numerical 53.73 mV/V
t_Si = 5 nm: formula 55.38 mV/V, numerical 55.38 mV/V


## Part 3 · Shuttle or dedicated

*Predict* the break-even area for a €3 M dedicated run.

In [4]:
print(f"break-even: {f.break_even_area(3e6, 17820):.0f} mm2")
areas = np.linspace(1, 250, 250)
plt.figure()
plt.plot(areas, [f.mpw_cost(a, 17820, 4) / 1e6 for a in areas], label="shuttle")
plt.axhline(3.0, color="0.4", ls="--", label="dedicated, EUR 3 M")
plt.xlabel("design area (mm2)"); plt.ylabel("cost (EUR M)"); plt.legend(); plt.grid(True)
plt.show()

break-even: 168 mm2


## Part 4 · Shrink economics

*Predict* whether a die with 60% analog and I/O gets cheaper.

In [5]:
a = f.migrated_area(4 * 2 / 3, 4 / 3, 6)
print(f"new area {a:.2f} mm2; cost ratio {1.8 * a / 10:.3f}")
fr = np.linspace(0.1, 0.8, 71)
ratio = [1.8 * f.migrated_area((1 - x) * 10 * 2 / 3, (1 - x) * 10 / 3, x * 10) / 10 for x in fr]
x_be = optimize.brentq(lambda x: 1.8 * f.migrated_area((1 - x) * 10 * 2 / 3, (1 - x) * 10 / 3, x * 10) / 10 - 1, 0.1, 0.8)
print(f"break-even analog fraction: {x_be:.2f}")
plt.figure()
plt.plot(fr, ratio); plt.axhline(1, color="0.5", ls=":")
plt.xlabel("analog and I/O fraction of 40 nm die"); plt.ylabel("die cost ratio, new/old"); plt.grid(True)
plt.show()

new area 6.42 mm2; cost ratio 1.156
break-even analog fraction: 0.42
